# 02 - Preprocessing - CSE-CIC-IDS2018

In [1]:
DATASET = "cic2018"
FT_ROOT = None

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import pandas as pd

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS, LABEL_COLUMN
from ids.preprocess.cleaning import clean
from ids.preprocess.encoding import fit_label_encoder, save_label_encoder, transform_labels

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "02_preprocess.log", "nb02")

with nbc.timed(log, "load raw_unified.parquet"):
    raw = pd.read_parquet(P.data / "raw_unified.parquet")
log.info("input: %d rows x %d cols", *raw.shape)

2026-07-07 05:35:22,345 INFO nb02: START load raw_unified.parquet


2026-07-07 05:35:23,028 INFO nb02: END   load raw_unified.parquet (0.7s)


2026-07-07 05:35:23,029 INFO nb02: input: 1999987 rows x 78 cols


## 1. Clean

In [3]:
with nbc.timed(log, "clean (inf->NaN, drop NaN rows, drop duplicates)"):
    clean_df, stats = clean(raw)
stats["pct_rows_dropped"] = round(100 * (1 - stats["output_rows"] / stats["input_rows"]), 3)
nbc.save_json(stats, P.data / "preprocess_stats.json")
log.info("cleaning: %s", stats)
stats

2026-07-07 05:35:23,046 INFO nb02: START clean (inf->NaN, drop NaN rows, drop duplicates)


2026-07-07 05:35:35,336 INFO ids.preprocess.cleaning: cleaning stats: {'input_rows': 1999987, 'replaced_inf': 14266, 'dropped_nan': 10408, 'dropped_duplicates': 402677, 'output_rows': 1586902}


2026-07-07 05:35:36,051 INFO nb02: END   clean (inf->NaN, drop NaN rows, drop duplicates) (13.0s)


2026-07-07 05:35:36,054 INFO nb02: cleaning: {'input_rows': 1999987, 'replaced_inf': 14266, 'dropped_nan': 10408, 'dropped_duplicates': 402677, 'output_rows': 1586902, 'pct_rows_dropped': 20.654}


{'input_rows': 1999987,
 'replaced_inf': 14266,
 'dropped_nan': 10408,
 'dropped_duplicates': 402677,
 'output_rows': 1586902,
 'pct_rows_dropped': 20.654}

### First and last 10 rows of the cleaned dataset

Quick visual check of the table the models will see (all 77 features + label).
Also saved to `data/clean_head_tail_preview.csv`.

In [4]:
preview = pd.concat([clean_df.head(10), clean_df.tail(10)])
preview.to_csv(P.data / "clean_head_tail_preview.csv")
log.info("saved clean_head_tail_preview.csv (%d rows)", len(preview))
with pd.option_context("display.max_columns", None):
    display(clean_df.head(10))
    display(clean_df.tail(10))

2026-07-07 05:35:36,078 INFO nb02: saved clean_head_tail_preview.csv (20 rows)


,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,bwd_pkt_len_mean,bwd_pkt_len_std,flow_byts_s,flow_pkts_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_tot,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_tot,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_len,bwd_header_len,fwd_pkts_s,bwd_pkts_s,pkt_len_min,pkt_len_max,pkt_len_mean,pkt_len_std,pkt_len_var,fin_flag_cnt,syn_flag_cnt,rst_flag_cnt,psh_flag_cnt,ack_flag_cnt,urg_flag_cnt,cwe_flag_count,ece_flag_cnt,down_up_ratio,pkt_size_avg,fwd_seg_size_avg,bwd_seg_size_avg,fwd_byts_b_avg,fwd_pkts_b_avg,fwd_blk_rate_avg,bwd_byts_b_avg,bwd_pkts_b_avg,bwd_blk_rate_avg,subflow_fwd_pkts,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,fwd_act_data_pkts,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,443.0,119402535.0,5.0,5.0,316.0,74.0,79.0,0.0,63.200000,35.329874,74.0,0.0,14.800000,33.093806,3.266262,0.083750,1.326695e+07,1.570966e+07,30018969.0,11558.0,119402535.0,2.985063e+07,3.595865e+05,30030527.0,29311254.0,119332607.0,2.983315e+07,3.945572e+05,30030552.0,29241316.0,1.0,0.0,0.0,0.0,100.0,100.0,0.041875,0.041875,0.0,79.0,42.636364,40.846720,1668.454545,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,46.900000,63.200000,14.800000,0.0,0.0,0.0,0.0,0.0,0.0,5.0,316.0,5.0,74.0,256.0,73.0,3.0,20.0,11568.25,1.198263e+01,11583.0,11558.0,2.982448e+07,388773.846809,30018969.0,29241316.0,Benign
1,445.0,61258.0,2.0,0.0,1.0,0.0,1.0,0.0,0.500000,0.707107,0.0,0.0,0.000000,0.000000,16.324398,32.648797,6.125800e+04,0.000000e+00,61258.0,61258.0,61258.0,6.125800e+04,0.000000e+00,61258.0,61258.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,52.0,0.0,32.648797,0.000000,0.0,1.0,0.666667,0.577350,0.333333,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.000000,0.500000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,253.0,-1.0,0.0,20.0,0.00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Benign
2,80.0,54267736.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.036854,5.426774e+07,0.000000e+00,54267736.0,54267736.0,54267736.0,5.426774e+07,0.000000e+00,54267736.0,54267736.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,0.036854,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,279.0,-1.0,0.0,20.0,0.00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Benign
3,3389.0,2405965.0,8.0,7.0,1144.0,1581.0,677.0,0.0,143.000000,227.969923,1173.0,0.0,225.857143,430.098604,1132.601680,6.234505,1.718546e+05,2.561095e+05,953160.0,88.0,2405965.0,3.437093e+05,3.609203e+05,1124415.0,297.0,2288593.0,3.814322e+05,2.836205e+05,953160.0,223210.0,0.0,0.0,0.0,0.0,172.0,152.0,3.325069,2.909436,0.0,1173.0,170.312500,319.402717,102018.095833,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,181.666667,143.000000,225.857143,0.0,0.0,0.0,0.0,0.0,0.0,8.0,1144.0,7.0,1581.0,8192.0,62856.0,5.0,20.0,0.00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Benign
4,53.0,294.0,1.0,1.0,43.0,59.0,43.0,43.0,43.000000,0.000000,59.0,59.0,59.000000,0.000000,346938.775510,6802.721088,2.940000e+02,0.000000e+00,294.0,294.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,8.0,8.0,3401.360544,3401.360544,43.0,59.0,48.333333,9.237604,85.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,72.500000,43.000000,59.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,43.0,1.0,59.0,-1.0,-1.0,0.0,8.0,0.00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,0.0,0.0,Benign
5,80.0,6293871.0,4.0,4.0,97.0,232.0,97.0,0.0,24.250000,48.500000,232.0,0.0,58.000000,116.000000,52.273076,1.271078,8.991244e+05,2.342833e+06,6211725.0,41.0,82146.0,2.738200e+04,4.713258e+04,81806.0,132.0,6293704.0,2.097901e+06,3.633217e+06,6293179.0,179.0,0

,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,bwd_pkt_len_mean,bwd_pkt_len_std,flow_byts_s,flow_pkts_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_tot,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_tot,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_len,bwd_header_len,fwd_pkts_s,bwd_pkts_s,pkt_len_min,pkt_len_max,pkt_len_mean,pkt_len_std,pkt_len_var,fin_flag_cnt,syn_flag_cnt,rst_flag_cnt,psh_flag_cnt,ack_flag_cnt,urg_flag_cnt,cwe_flag_count,ece_flag_cnt,down_up_ratio,pkt_size_avg,fwd_seg_size_avg,bwd_seg_size_avg,fwd_byts_b_avg,fwd_pkts_b_avg,fwd_blk_rate_avg,bwd_byts_b_avg,bwd_pkts_b_avg,bwd_blk_rate_avg,subflow_fwd_pkts,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,fwd_act_data_pkts,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
1586892,443.0,503.0,3.0,0.0,77.0,0.0,46.0,0.0,25.666667,23.459184,0.0,0.0,0.000000,0.000000,153081.510900,5964.214712,2.515000e+02,3.288047e+02,484.0,19.0,503.0,2.515000e+02,3.288047e+02,484.0,19.0,0.0,0.0000,0.0000,0.0,0.0,1.0,0.0,0.0,0.0,60.0,0.0,5964.214712,0.000000,0.0,46.0,30.750000,21.685248,470.25000,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,41.000000,25.666667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,3.0,77.0,0.0,0.0,256.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1586893,30927.0,86221456.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.023196,8.620000e+07,0.000000e+00,86200000.0,86200000.0,86200000.0,8.620000e+07,0.000000e+00,86200000.0,86200000.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,0.023196,0.000000,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,64000.0,-1.0,0.0,20.0,0.0,0.0,0.0,0.0,86200000.0,0.0,86200000.0,86200000.0,Benign
1586894,3389.0,1773257.0,9.0,7.0,1144.0,1581.0,677.0,0.0,127.111111,218.508835,1173.0,0.0,225.857143,430.098604,1536.720284,9.022945,1.182171e+05,2.342736e+05,937544.0,95.0,1773257.0,2.216571e+05,3.613533e+05,1105165.0,6487.0,1670032.0,278338.6667,323944.9991,937544.0,120606.0,0.0,0.0,0.0,0.0,192.0,152.0,5.075406,3.947538,0.0,1173.0,160.294118,312.006764,97348.22059,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,170.312500,127.111111,225.857143,0.0,0.0,0.0,0.0,0.0,0.0,9.0,1144.0,7.0,1581.0,8192.0,62856.0,5.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1586895,42808.0,84241018.0,2.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.023741,8.420000e+07,0.000000e+00,84200000.0,84200000.0,84200000.0,8.420000e+07,0.000000e+00,84200000.0,84200000.0,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,0.023741,0.000000,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,62566.0,-1.0,0.0,20.0,0.0,0.0,0.0,0.0,84200000.0,0.0,84200000.0,84200000.0,Benign
1586896,3389.0,4130458.0,14.0,8.0,1441.0,1731.0,725.0,0.0,102.928571,191.148941,1179.0,0.0,216.375000,405.464791,767.953578,5.326286,1.966885e+05,2.614885e+05,954461.0,3.0,4130458.0,3.177275e+05,3.612876e+05,1301037.0,3.0,4057527.0,579646.7143,371677.3574,1025743.0,108079.0,0.0,0.0,0.0,0.0,292.0,172.0,3.389455,1.936831,0.0,1179.0,137.913043,278.894720,77782.26482,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,144.181818,102.928571,216.375000,0.0,0.0,0.0,0.0,0.0,0.0,14.0,1441.0,8.0,1731.0,8192.0,62644.0,7.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1586897,51703.0,66212833.0,9.0,0.0,495.0,0.0,58.0,31.0,55.000000,9.000000,0.0,0.0,0.000000,0.000000,7.475892,0.135925,8.276604e+06,1.140000e+07,32800000.0,257936.0,66200000.0,8.276604e+06,1.140000e+07,32800000.0,257936.0,0.0,0.0000,0.0000,0.0,0.0,1.0,0.0,0.0,0.0,180.0,0.0,0.135925,0.000000,31.0,58.0,52.600000,11.384200,129.60000,0.0,1.0,0.0,0.0,1.0

## 2. Encode labels

In [5]:
encoder = fit_label_encoder(clean_df[LABEL_COLUMN])
save_label_encoder(encoder, P.data / "label_encoder.joblib")
y = transform_labels(encoder, clean_df[LABEL_COLUMN])
mapping = {cls: int(i) for i, cls in enumerate(encoder.classes_)}
nbc.save_json(mapping, P.data / "label_mapping.json")
log.info("%d classes encoded: %s", len(mapping), list(mapping)[:8] + ["..."])
mapping

2026-07-07 05:35:36,404 INFO nb02: 15 classes encoded: ['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye', '...']


{'Benign': 0,
 'Bot': 1,
 'Brute Force -Web': 2,
 'Brute Force -XSS': 3,
 'DDOS attack-HOIC': 4,
 'DDOS attack-LOIC-UDP': 5,
 'DDoS attacks-LOIC-HTTP': 6,
 'DoS attacks-GoldenEye': 7,
 'DoS attacks-Hulk': 8,
 'DoS attacks-SlowHTTPTest': 9,
 'DoS attacks-Slowloris': 10,
 'FTP-BruteForce': 11,
 'Infilteration': 12,
 'SQL Injection': 13,
 'SSH-Bruteforce': 14}

## 3. Save the clean modeling table + final class distribution

In [6]:
out = clean_df[CANONICAL_COLUMNS].copy()
out["y"] = y
out.to_parquet(P.data / "clean.parquet", index=False)

final_dist = (clean_df[LABEL_COLUMN].value_counts()
              .rename_axis("label").reset_index(name="count"))
final_dist["pct"] = (100 * final_dist["count"] / len(clean_df)).round(4)
final_dist.to_csv(P.data / "final_class_distribution.csv", index=False)
log.info("saved clean.parquet: %d rows x %d features (+y) | %d classes",
         len(out), len(CANONICAL_COLUMNS), final_dist.shape[0])
final_dist

2026-07-07 05:35:40,436 INFO nb02: saved clean.parquet: 1586902 rows x 77 features (+y) | 15 classes


,label,count,pct
0,Benign,1341112,84.5113
1,Infilteration,70445,4.4392
2,DDOS attack-HOIC,56112,3.5359
3,DoS attacks-Hulk,47929,3.0203
4,Bot,28272,1.7816
5,SSH-Bruteforce,18062,1.1382
6,DDoS attacks-LOIC-HTTP,14498,0.9136
7,DoS attacks-GoldenEye,7911,0.4985
8,DoS attacks-Slowloris,1962,0.1236
9,DDOS attack-LOIC-UDP,330,0.0208
